# Group Knapsack Game (GKG) — Exp 1 & Exp 2

**Exp 1**: Best PNE and POS (BRD + GZR @ alpha=1)
- BRD warm-starts GZR (if PNE found); GZR optimises to MIP gap=0 (`stop_at_first=False`) to find the **best** (max social welfare) PNE
- SO = social optimum (max welfare without equilibrium constraints)
- POS = SO / best_PNE (≥ 1)

**Exp 2**: Tightest Alpha Search (TAS) for non-PNE instances
- Uses `stop_at_first=True` (only needs existence, not optimality)

In [ ]:
from pathlib import Path
import sys
import json
import time
from dataclasses import replace

import numpy as np
import pandas as pd

# Ensure we can import the `gkg` package
    



In [ ]:
from gipg.gkg.instance import GKGInstance
from gipg.gkg.best_response import solve_best_response
from gipg.gkg.heuristics import brd_random_restart, alpha_of_profile
from gipg.gkg.gzr import solve_gzr
from gipg.gkg.social_optimum import solve_social_optimum, compute_pos

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('GKG modules loaded successfully.')

## 1. Load Instances

In [ ]:
INST_DIR = Path('../data/gkg') / 'gkg_instances'
assert INST_DIR.exists(), f'Missing instance folder: {INST_DIR}'

instances = []
for fp in sorted(INST_DIR.glob('*.json')):
    with open(fp, 'r', encoding='utf-8') as f:
        d = json.load(f)
    inst = GKGInstance.from_dict(d)
    new_meta = dict(getattr(inst, 'meta', {}) or {})
    new_meta['file'] = fp.name
    inst = replace(inst, meta=new_meta)
    instances.append(inst)

print(f'Loaded {len(instances)} instances from {INST_DIR}')

# Summary
print('\n--- By (n, m) ---')
from collections import Counter
size_counts = Counter()
for inst in instances:
    size_counts[(inst.n, inst.m)] += 1
for k in sorted(size_counts.keys()):
    print(f'  n={k[0]}, m={k[1]}: {size_counts[k]} instances')

print('\n--- By (cap_factor, corr) ---')
cf_counts = Counter()
for inst in instances:
    cf_counts[(inst.cap_factor, inst.corr)] += 1
for k in sorted(cf_counts.keys()):
    print(f'  cf={k[0]}, corr={k[1]}: {cf_counts[k]} instances')

## 2. Experiment 1: Best PNE and POS (BRD + GZR @ alpha=1)

For each instance:
1. **BRD** warm-starts **GZR @ alpha=1** (if BRD found a PNE)
2. **GZR** with `stop_at_first=False` — optimises to MIP gap=0, finding the **best** PNE (max welfare)
3. **Social Optimum** computed separately
4. **POS** = SO / best_PNE

In [ ]:
TOTAL_TIME_LIMIT = 600.0  # Total budget for BRD + GZR combined
SO_TIME_LIMIT = 600.0     # 10 minutes for social optimum

results_exp1 = []

for idx, inst in enumerate(instances):
    tag = inst.meta.get('file', f'inst_{idx}')

    row = {
        'tag': tag,
        'n': inst.n,
        'm': inst.m,
        'cap_factor': inst.cap_factor,
        'corr': inst.corr,
        'seed': inst.seed,
    }

    # --- Phase 1: BRD ---
    try:
        x_brd, brd_pne, brd_time = brd_random_restart(
            inst, max_init=3, max_round=15, seed=0,
        )
        row['brd_found_pne'] = brd_pne
        row['brd_time'] = brd_time
        row['brd_alpha'] = alpha_of_profile(inst, x_brd) if brd_pne else float('inf')
        if brd_pne:
            row['initial_pne_welfare'] = float(sum(
                int(inst.p[i, j]) * int(x_brd[i, j])
                for i in range(inst.n) for j in range(inst.m)
            ))
    except Exception as e:
        row['brd_found_pne'] = False
        row['brd_time'] = 0.0
        x_brd = None
        brd_pne = False
        brd_time = 0.0
        print(f'  BRD ERROR: {e}')

    # --- Phase 2: GZR @ alpha=1, warm-started with BRD profile ---
    gzr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)

    warm = x_brd if brd_pne else None
    try:
        gzr_res = solve_gzr(
            inst, alpha=1.0, time_limit=gzr_time_limit,
            warm_start=warm,
            stop_at_first=False, verbose=False,
        )
        row['gzr_status'] = gzr_res.status
        row['gzr_mip_gap'] = gzr_res.mip_gap
        row['gzr_obj_bound'] = gzr_res.obj_bound
        row['gzr_cuts'] = gzr_res.cuts_added
        row['gzr_br_calls'] = gzr_res.br_calls
        row['gzr_time'] = brd_time + gzr_res.runtime   # BRD + GZR combined
        row['gzr_obj_val'] = gzr_res.obj_val
        row['gzr_first_pne_time'] = (brd_time + gzr_res.first_pne_time
                                            if gzr_res.first_pne_time is not None else None)
    except Exception as e:
        gzr_res = None
        row['gzr_status'] = 'ERROR'
        row['gzr_mip_gap'] = None
        row['gzr_obj_bound'] = None
        row['gzr_cuts'] = None
        row['gzr_time'] = None
        row['gzr_first_pne_time'] = None
        print(f'  GZR ERROR: {e}')

    # Best PNE welfare
    if gzr_res is not None and gzr_res.profile is not None:
        row['best_pne_welfare'] = float(sum(
            int(inst.p[i, j]) * int(gzr_res.profile[i, j])
            for i in range(inst.n) for j in range(inst.m)
        ))
    elif brd_pne:
        row['best_pne_welfare'] = row.get('initial_pne_welfare')
    else:
        row['best_pne_welfare'] = None

    # --- Phase 3: Social Optimum ---
    try:
        so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
        row['so_status'] = so_res.status
        row['so_welfare'] = so_res.opt_cost
        row['so_time'] = so_res.runtime
    except Exception as e:
        row['so_status'] = 'ERROR'
        row['so_welfare'] = None
        row['so_time'] = None
        print(f'  SO ERROR: {e}')

    # --- POS ---
    row['pos'] = None
    if row.get('best_pne_welfare') is not None and row.get('so_welfare') is not None and row['best_pne_welfare'] > 0:
        row['pos'] = compute_pos(row['so_welfare'], row['best_pne_welfare'])

    results_exp1.append(row)

    # Progress
    pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
    gzr_time_str = f'{row["gzr_time"]:.1f}s' if row.get('gzr_time') is not None else '?s'
    gzr_cuts_str = f'{row["gzr_cuts"]}cuts' if row.get('gzr_cuts') is not None else '?cuts'
    pos_str = f'POS={row["pos"]:.3f}' if row['pos'] is not None else 'POS=N/A'
    fpne_str = f'1stPNE={row["gzr_first_pne_time"]:.1f}s' if row.get('gzr_first_pne_time') is not None else '1stPNE=N/A'
    if (idx + 1) % 30 == 0 or idx == len(instances) - 1:
        print(f'[{idx+1}/{len(instances)}] {tag}: {pne_str} | GZR {row.get("gzr_status","?")} '
              f'({gzr_time_str}, {gzr_cuts_str}) | {pos_str} | {fpne_str}')

df_exp1 = pd.DataFrame(results_exp1)
df_exp1.to_csv(RESULTS_DIR / 'gkg_exp1_brd_gzr_pos.csv', index=False)
print(f'\nSaved {len(df_exp1)} rows to gkg_exp1_brd_gzr_pos.csv')

In [ ]:
# Experiment 1 Summary
print('=== Experiment 1 Summary ===')

n_total = len(df_exp1)
n_brd_pne = df_exp1['brd_found_pne'].sum()
n_gzr_opt = (df_exp1['gzr_status'] == 'OPTIMAL').sum()
n_gzr_inf = (df_exp1['gzr_status'] == 'INFEASIBLE').sum()
n_gzr_tl = (df_exp1['gzr_status'] == 'TIME_LIMIT').sum()
n_pos = df_exp1['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'GZR OPTIMAL: {n_gzr_opt} ({n_gzr_opt/n_total:.1%})')
print(f'GZR INFEASIBLE: {n_gzr_inf} ({n_gzr_inf/n_total:.1%})')
print(f'GZR TIME_LIMIT: {n_gzr_tl} ({n_gzr_tl/n_total:.1%})')
print(f'POS computed: {n_pos} ({n_pos/n_total:.1%})')

# By (n, m)
print('\n--- By (n, m) ---')
summary = df_exp1.groupby(['n', 'm']).agg(
    count=('tag', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary)

# By (cap_factor, corr)
print('\n--- By (cap_factor, corr) ---')
summary_cf = df_exp1.groupby(['cap_factor', 'corr']).agg(
    count=('tag', 'count'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary_cf)

## 2B. Experiment 1B: GZR Only (No Warm Start) — Comparison with Duguet

GZR @ alpha=1 with `stop_at_first=True`, **no BRD warm start**, 10-minute time limit.
This isolates the performance of our CEI-GZR branch-and-cut solver for direct comparison
with Duguet et al.'s intersection cut approach (Table 1). The goal is to find **any** PNE.

In [ ]:
GZR_ONLY_TIME_LIMIT = 600.0  # 10 minutes

results_exp1b = []

for idx, inst in enumerate(instances):
    tag = inst.meta.get('file', f'inst_{idx}')

    row = {
        'tag': tag,
        'n': inst.n,
        'm': inst.m,
        'cap_factor': inst.cap_factor,
        'corr': inst.corr,
        'seed': inst.seed,
    }

    # --- GZR @ alpha=1, NO warm start, stop at first PNE ---
    try:
        gzr_res = solve_gzr(
            inst, alpha=1.0, time_limit=GZR_ONLY_TIME_LIMIT,
            warm_start=None,
            stop_at_first=True, verbose=False,
        )
        row['gzr_status'] = gzr_res.status
        row['gzr_cuts'] = gzr_res.cuts_added
        row['gzr_br_calls'] = gzr_res.br_calls
        row['gzr_time'] = gzr_res.runtime
        row['gzr_obj_val'] = gzr_res.obj_val
        row['gzr_first_pne_time'] = gzr_res.first_pne_time
    except Exception as e:
        gzr_res = None
        row['gzr_status'] = 'ERROR'
        row['gzr_cuts'] = None
        row['gzr_br_calls'] = None
        row['gzr_time'] = None
        row['gzr_obj_val'] = None
        row['gzr_first_pne_time'] = None
        print(f'  GZR ERROR ({tag}): {e}')

    # PNE welfare
    if gzr_res is not None and gzr_res.profile is not None:
        row['pne_welfare'] = float(sum(
            int(inst.p[i, j]) * int(gzr_res.profile[i, j])
            for i in range(inst.n) for j in range(inst.m)
        ))
    else:
        row['pne_welfare'] = None

    results_exp1b.append(row)

    # Progress
    gzr_time_str = f'{row["gzr_time"]:.1f}s' if row.get('gzr_time') is not None else '?s'
    gzr_cuts_str = f'{row["gzr_cuts"]}cuts' if row.get('gzr_cuts') is not None else '?cuts'
    fpne_str = f'1stPNE={row["gzr_first_pne_time"]:.1f}s' if row.get('gzr_first_pne_time') is not None else '1stPNE=N/A'
    if (idx + 1) % 30 == 0 or idx == len(instances) - 1:
        print(f'[{idx+1}/{len(instances)}] {tag}: GZR {row.get("gzr_status","?")} '
              f'({gzr_time_str}, {gzr_cuts_str}) | {fpne_str}')

df_exp1b = pd.DataFrame(results_exp1b)
df_exp1b.to_csv(RESULTS_DIR / 'gkg_exp1b_gzr_only.csv', index=False)
print(f'\nSaved {len(df_exp1b)} rows to gkg_exp1b_gzr_only.csv')

In [ ]:
# Experiment 1B Summary
print('=== Experiment 1B Summary (GZR Only, No Warm Start, stop_at_first=True) ===')

n_total = len(df_exp1b)
n_gzr_feas = (df_exp1b['gzr_status'] == 'FEASIBLE').sum()
n_gzr_opt = (df_exp1b['gzr_status'] == 'OPTIMAL').sum()
n_gzr_inf = (df_exp1b['gzr_status'] == 'INFEASIBLE').sum()
n_gzr_tl = (df_exp1b['gzr_status'] == 'TIME_LIMIT').sum()
n_found = n_gzr_feas + n_gzr_opt

print(f'Total instances: {n_total}')
print(f'PNE found (FEASIBLE+OPTIMAL): {n_found} ({n_found/n_total:.1%})')
print(f'  FEASIBLE (stopped at first): {n_gzr_feas}')
print(f'  OPTIMAL (proved no PNE / exhausted): {n_gzr_opt}')
print(f'GZR INFEASIBLE: {n_gzr_inf} ({n_gzr_inf/n_total:.1%})')
print(f'GZR TIME_LIMIT: {n_gzr_tl} ({n_gzr_tl/n_total:.1%})')

# By (n, m) — matches Duguet Table 1 layout
print('\n--- By (n, m) ---')
summary_1b = df_exp1b.groupby(['n', 'm']).agg(
    count=('tag', 'count'),
    n_found=('gzr_status', lambda x: ((x == 'FEASIBLE') | (x == 'OPTIMAL')).sum()),
    n_tl=('gzr_status', lambda x: (x == 'TIME_LIMIT').sum()),
    avg_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_br_calls=('gzr_br_calls', 'mean'),
    avg_first_pne=('gzr_first_pne_time', 'mean'),
).round(4)
print(summary_1b)

# By (cap_factor, corr)
print('\n--- By (cap_factor, corr) ---')
summary_1b_cf = df_exp1b.groupby(['cap_factor', 'corr']).agg(
    count=('tag', 'count'),
    n_found=('gzr_status', lambda x: ((x == 'FEASIBLE') | (x == 'OPTIMAL')).sum()),
    n_tl=('gzr_status', lambda x: (x == 'TIME_LIMIT').sum()),
    avg_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
).round(4)
print(summary_1b_cf)

# By (n, m, cap_factor, corr) — full breakdown for Table 1 comparison
print('\n--- Full breakdown: (n, m, cap_factor, corr) ---')
full_summary = df_exp1b.groupby(['n', 'm', 'cap_factor', 'corr']).agg(
    count=('tag', 'count'),
    n_found=('gzr_status', lambda x: ((x == 'FEASIBLE') | (x == 'OPTIMAL')).sum()),
    n_tl=('gzr_status', lambda x: (x == 'TIME_LIMIT').sum()),
    avg_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_br_calls=('gzr_br_calls', 'mean'),
).round(4)
pd.set_option('display.max_rows', 200)
print(full_summary)

## 3. Experiment 2: Tightest Alpha Search (TAS) for Non-PNE Instances

Only instances where Exp 1 GZR reported **INFEASIBLE** or **TIME_LIMIT** (and BRD found no PNE).
Bisection with verified/unverified bounds.

In [ ]:
TAS_GZR_LIMIT = 1200.0   # 20 minutes per GZR call in TAS
TAS_EPS = 1e-2
TAS_MAX_ITERS = 30

# Gather failed instances from Exp1
failed_indices = [
    idx for idx, r in enumerate(results_exp1)
    if r['gzr_status'] in ('INFEASIBLE', 'TIME_LIMIT') and not r.get('brd_found_pne', False)
]
print(f'Exp2 candidates: {len(failed_indices)} instances')

results_exp2 = []

for count, idx in enumerate(failed_indices):
    inst = instances[idx]
    tag = inst.meta.get('file', f'inst_{idx}')
    start_time = time.time()

    row = {
        'tag': tag,
        'n': inst.n,
        'm': inst.m,
        'cap_factor': inst.cap_factor,
        'corr': inst.corr,
        'seed': inst.seed,
    }

    # --- Phase 0: BRD upper bound on alpha ---
    x_brd, brd_pne, _ = brd_random_restart(
        inst, max_init=5, max_round=20, seed=0,
    )
    alpha_init = alpha_of_profile(inst, x_brd)
    ub_verified = alpha_init
    best_profile = x_brd

    # --- Phase 1: Reuse Exp1 result for alpha=1 ---
    exp1_row = results_exp1[idx]
    if exp1_row['gzr_status'] == 'INFEASIBLE':
        lb_verified = 1.0
        lb_unverified = 1.0
    else:  # TIME_LIMIT
        lb_verified = None
        lb_unverified = 1.0

    # --- Phase 2: Bisection ---
    n_iters = 0
    history = []

    while (ub_verified - lb_unverified > TAS_EPS) and (n_iters < TAS_MAX_ITERS):
        n_iters += 1
        alpha_mid = 0.5 * (lb_unverified + ub_verified)

        gzr_mid = solve_gzr(
            inst, alpha=alpha_mid, time_limit=TAS_GZR_LIMIT,
            stop_at_first=True, verbose=False,
        )

        if gzr_mid.status in ('FEASIBLE', 'OPTIMAL') and gzr_mid.profile is not None:
            ub_verified = alpha_mid
            best_profile = gzr_mid.profile
            bound_type = 'verified_ub'
        elif gzr_mid.status == 'INFEASIBLE':
            lb_unverified = alpha_mid
            lb_verified = alpha_mid if lb_verified is None else max(lb_verified, alpha_mid)
            bound_type = 'verified_lb'
        else:
            lb_unverified = alpha_mid
            bound_type = 'unverified_lb'

        history.append({
            'iter': n_iters, 'alpha_mid': alpha_mid,
            'status': gzr_mid.status, 'bound_type': bound_type,
            'runtime': gzr_mid.runtime,
        })
        print(f'  iter {n_iters}: alpha={alpha_mid:.4f} -> {gzr_mid.status} ({bound_type}) '
              f'[{gzr_mid.runtime:.1f}s]')

    row['alpha_init'] = alpha_init
    row['alpha_star'] = ub_verified
    row['alpha_lb_verified'] = lb_verified
    row['alpha_lb_unverified'] = lb_unverified
    row['alpha_ub_verified'] = ub_verified
    row['n_bisect_iters'] = n_iters
    row['tas_time_total'] = time.time() - start_time
    results_exp2.append(row)

    print(f'[{count+1}/{len(failed_indices)}] {tag}: alpha*={ub_verified:.4f} '
          f'(lb_v={lb_verified}, lb_u={lb_unverified:.4f}) '
          f'{n_iters} iters, {row["tas_time_total"]:.1f}s')

df_exp2 = pd.DataFrame(results_exp2)
if len(df_exp2) > 0:
    df_exp2.to_csv(RESULTS_DIR / 'gkg_exp2_tas.csv', index=False)
    print(f'\nSaved {len(df_exp2)} rows to gkg_exp2_tas.csv')
else:
    print('\nNo instances needed TAS (all solved in Exp1).')

In [ ]:
# Experiment 2 Summary
print('=== Experiment 2: TAS Summary ===')
if len(df_exp2) > 0:
    print(f'Instances in TAS: {len(df_exp2)}')
    print(f'Mean alpha*: {df_exp2["alpha_star"].mean():.4f}')
    print(f'Max  alpha*: {df_exp2["alpha_star"].max():.4f}')
    n_verified_lb = df_exp2['alpha_lb_verified'].notna().sum()
    print(f'Verified lower bound: {n_verified_lb} ({n_verified_lb/len(df_exp2):.1%})')
    print(f'Mean bisection iters: {df_exp2["n_bisect_iters"].mean():.1f}')

    print('\n--- By (n, m) ---')
    summary2 = df_exp2.groupby(['n', 'm']).agg(
        count=('tag', 'count'),
        avg_alpha_star=('alpha_star', 'mean'),
        max_alpha_star=('alpha_star', 'max'),
        avg_iters=('n_bisect_iters', 'mean'),
    ).round(4)
    print(summary2)
else:
    print('No instances required TAS.')

## 4. Combined Results

In [ ]:
# Merge Exp1 and Exp2
if len(df_exp2) > 0:
    exp2_cols = ['tag', 'alpha_init', 'alpha_star', 'alpha_lb_verified',
                 'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters']
    df_all = pd.merge(df_exp1, df_exp2[exp2_cols], on='tag', how='left')
else:
    df_all = df_exp1.copy()
    for col in ['alpha_init', 'alpha_star', 'alpha_lb_verified',
                'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters']:
        df_all[col] = np.nan

# For instances solved in Exp1, alpha_star = 1.0
mask_solved = (df_all['gzr_status'] == 'OPTIMAL') | (df_all['brd_found_pne'] == True)
df_all.loc[mask_solved & df_all['alpha_star'].isna(), 'alpha_star'] = 1.0

df_all.to_csv(RESULTS_DIR / 'gkg_results_all.csv', index=False)
print(f'Combined results: {len(df_all)} rows -> gkg_results_all.csv')
print(f'Columns: {list(df_all.columns)}')

# Final summary
print('\n=== Final Summary ===')
for n_val in sorted(df_all['n'].unique()):
    sub = df_all[df_all['n'] == n_val]
    n_pne = ((sub['gzr_status'] == 'OPTIMAL') | (sub['brd_found_pne'] == True)).sum()
    pos_ok = sub['pos'].notna()
    pos_str = f'POS mean={sub.loc[pos_ok, "pos"].mean():.3f}' if pos_ok.any() else 'no POS'
    tas_sub = sub[sub['n_bisect_iters'].notna() & (sub['n_bisect_iters'] > 0)]
    tas_str = f'TAS: {len(tas_sub)} inst' if len(tas_sub) > 0 else 'no TAS'
    print(f'  n={n_val}: {len(sub)} inst, PNE={n_pne}, {pos_str}, {tas_str}')